## 쿨루프 시공 대상 여부 분류
- Tasks
    1. Data Preprocessing
        - 모델 사용을 위해 데이터를 일관성 있게 정리
    2. Object Detection
        - 전처리 된 데이터를 이용하여 학습을 진행
        - 인공위성 지도를 이용하여 이미지를 **추가적으로** 10장 수집하고 추론 과정에서 사용

### 데이터셋 다운로드 및 압축 해제
- cool_roof_image.zip : 이미지 데이터 압축 파일
- cool_roof_yolo_labels.zip : YOLO 모델 형식에 맞게 가공된 레이블 파일

In [1]:
%pip install ultralytics
%pip install -q ultralytics roboflow python-dotenv
%pip install roboflow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



In [2]:
import os
import zipfile
import os, glob, shutil
from tqdm import tqdm
import random

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

### 환경 설정 — API 키 관리

- Roboflow API 키는 코드에 하드코딩하지 않고 `.env` 파일로 분리해 환경변수로 주입 
- `.env`는 `.gitignore`에 등록되어 저장소에 포함되지 않으며,
- 재현 시에는 `.env.example`을 복사해 본인 키를 입력

In [3]:
from pathlib import Path

# 실행 환경 자동 감지 — google.colab 모듈 존재 여부로 판별
try:
    from google.colab import userdata, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# API 키 로딩 — 코랩은 Secrets, 로컬은 .env (키를 코드·저장소에 남기지 않는다)
if IN_COLAB:
    api_key = userdata.get('ROBOFLOW_API_KEY')
else:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    api_key = os.getenv('ROBOFLOW_API_KEY')

if api_key is None:
    raise ValueError("ROBOFLOW_API_KEY 없음 — 코랩: Secrets 등록 / 로컬: .env 확인")

print(f"환경: {'Colab' if IN_COLAB else 'Local'}")

환경: Local


In [4]:
# 경로를 한 곳에서 정의 — 이후 모든 셀은 이 상수만 참조
if IN_COLAB:
    drive.mount('/content/drive')
    BASE = Path('/content')
    DATA_SRC = Path('/content/drive/MyDrive/deep_see_mini4-2')  # zip 위치
else:
    local_path = os.getenv('LOCAL_PATH')
    BASE = Path('.')                    # 프로젝트 루트에서 노트북 실행 기준
    DATA_SRC = Path(local_path)       # 로컬 zip 위치 (gitignore 대상)

DATASETS = BASE / 'datasets'

In [5]:
from roboflow import Roboflow

# Roboflow 데이터셋 다운로드 — api_key는 '환경 설정' 섹션에서 로딩됨
rf = Roboflow(api_key=api_key)
project = rf.workspace("aivle-school-iwkpi").project("roof-ls63v")
dataset = project.version(1).download("yolov8-obb")

loading Roboflow workspace...
loading Roboflow project...


In [6]:
def dataset_extract(zip_path, out_name):
    """zip을 BASE/out_name 폴더에 해제 (이미 있으면 건너뜀)"""
    out_dir = BASE / out_name
    if out_dir.exists():
        print(f'{out_name}: 이미 존재, 건너뜀')
        return
    with zipfile.ZipFile(zip_path) as z:
        for f in tqdm(z.namelist(), desc=f'Extracting {out_name}'):
            z.extract(f, out_dir)

dataset_extract(DATA_SRC / 'cool_roof_images.zip', 'cool_roof_images')
dataset_extract(DATA_SRC / 'cool_roof_yolo_labels.zip', 'cool_roof_labels')

cool_roof_images: 이미 존재, 건너뜀
cool_roof_labels: 이미 존재, 건너뜀


### 폴더 생성
- YOLO 모델에서 요구하는 폴더의 형식 맞추기
- 해당 형식에 맞춰 폴더 생성

In [7]:
# YOLO 요구 폴더 구조 생성 — 경로는 상수만 참조 (코랩/로컬 공통)
for split in ['train', 'val', 'test']:
    for sub in ['images', 'labels']:
        (DATASETS / split / sub).mkdir(parents=True, exist_ok=True)

### 데이터 스플릿 & 파일 이동
1. 재현을 위한 난수 고정 : 2024
2. 테스트셋 데이터 20%
- 위 설정에 맞게 데이터를 나누고, 위 과정에서 생성한 폴더에 이동

In [8]:
# 기본 데이터 (제공 200장) — 이미지·라벨 쌍 로딩
image_files = sorted(glob.glob(str(BASE / 'cool_roof_images' / '*.jpg')))
label_files = sorted(glob.glob(str(BASE / 'cool_roof_labels' / 'obj_train_data' / '*.txt')))

# 빈 glob이 조용히 통과하는 것을 조기 차단
print(f"이미지 {len(image_files)}장 / 라벨 {len(label_files)}개")
assert len(image_files) == len(label_files) > 0, "glob 결과 확인 — 경로 오류 또는 쌍 불일치"

data = list(zip(image_files, label_files))
train_data, val_data = train_test_split(data, test_size=0.2, random_state=2024)
train_files, train_labels = zip(*train_data)
val_files, val_labels = zip(*val_data)

이미지 200장 / 라벨 200개


In [9]:
for img, lbl in zip(train_files, train_labels):
    shutil.copy2(img, DATASETS / 'train' / 'images')
    shutil.copy2(lbl, DATASETS / 'train' / 'labels')

for img, lbl in zip(val_files, val_labels):
    shutil.copy2(img, DATASETS / 'val' / 'images')
    shutil.copy2(lbl, DATASETS / 'val' / 'labels')

In [10]:
#  Roboflow roof-1 병합 — 기록된 339장 학습 재현용
rf_dir = Path(dataset.location)   # version.download()가 반환한 폴더
for sub_from, sub_to in [('train', 'train'), ('valid', 'val')]:
    for f in (rf_dir / sub_from / 'images').glob('*'):
        shutil.copy2(f, DATASETS / sub_to / 'images')
    for f in (rf_dir / sub_from / 'labels').glob('*'):
        shutil.copy2(f, DATASETS / sub_to / 'labels')

In [11]:
# 직접 수집한 위성 이미지(Roboflow test 스플릿)를 추론용 폴더로 복사
for f in (rf_dir / 'test' / 'images').glob('*'):
    shutil.copy2(f, DATASETS / 'test' / 'images')

### YOLO 모델에 적용할 YAML 생성하기
- 지붕에는 두 가지 형태가 존재, 클래스 구분에 주의
- cool roof
- generic roof

In [12]:
import yaml

data = {
    "path": str(DATASETS.resolve()),   # 데이터셋 루트를 절대경로로 명시 — 해석 규칙에 안 휘둘림
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: 'cool roof', 1: 'generic roof'},
}

yaml_path = BASE / 'tld.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(yaml.safe_load(open(yaml_path)))   # path가 올바른 절대경로인지 눈으로 확인

{'names': {0: 'cool roof', 1: 'generic roof'}, 'path': 'C:\\Users\\sdc02\\aivle_school\\deep_see_mini4-2\\datasets', 'test': 'test/images', 'train': 'train/images', 'val': 'val/images'}


### YOLO v8 모델
- yaml 파일의 경로 설정

In [13]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt', task='detect')

model.train(data=str(yaml_path), epochs=20, patience=5, save=True, name='train', exist_ok=True)    # 항상 runs/detect/train 폴더를 재사용

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cpu CPU (AMD Ryzen 5 5600X 6-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=tld.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-7, nbs=64, nms=False, opset=None, optim

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000265DAD87450>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [14]:
for d in sorted((BASE / 'runs' / 'detect').iterdir()):
    has_best = (d / 'weights' / 'best.pt').exists()
    print(f"{d.name:12s} → best.pt {'있음 ✅' if has_best else '없음 (실패 잔해)'}")

predict      → best.pt 없음 (실패 잔해)
train        → best.pt 없음 (실패 잔해)
train-2      → best.pt 없음 (실패 잔해)
train-3      → best.pt 없음 (실패 잔해)
train-4      → best.pt 없음 (실패 잔해)
train-5      → best.pt 없음 (실패 잔해)
train-6      → best.pt 있음 ✅
train-7      → best.pt 있음 ✅


In [15]:
from ultralytics import YOLO

# 가장 최근 학습 결과의 best.pt 자동 탐색 (실패 잔해 폴더는 자동 제외)
runs_root = BASE / 'runs' / 'detect'
candidates = [d for d in runs_root.glob('train*') if (d / 'weights' / 'best.pt').exists()]
assert candidates, "학습 결과 없음 — 4. 학습 섹션을 먼저 실행하세요"

latest = max(candidates, key=lambda d: (d / 'weights' / 'best.pt').stat().st_mtime)
weights = latest / 'weights' / 'best.pt'
print(f"사용할 가중치: {weights}")

model = YOLO(weights)
results = model.predict(source=str(DATASETS / 'test' / 'images'), save=True)

사용할 가중치: runs\detect\train-7\weights\best.pt

image 1/15 C:\Users\sdc02\aivle_school\deep_see_mini4-2\datasets\test\images\-2024-04-12-102034_png.rf.43a79b22a950d39647a7bd7660c26ef0.jpg: 640x640 5 generic roofs, 52.3ms
image 2/15 C:\Users\sdc02\aivle_school\deep_see_mini4-2\datasets\test\images\-2024-04-12-103713_png.rf.1ceb01b1a9974d635083a18a7e393563.jpg: 640x640 2 cool roofs, 5 generic roofs, 42.2ms
image 3/15 C:\Users\sdc02\aivle_school\deep_see_mini4-2\datasets\test\images\-2024-04-12-104005_png.rf.1b562151d5d7341afb88a1a10302bb38.jpg: 640x640 9 generic roofs, 46.1ms
image 4/15 C:\Users\sdc02\aivle_school\deep_see_mini4-2\datasets\test\images\-2024-04-12-105825_png.rf.8c0b883a9c4ee454a3d278fe9c790d9f.jpg: 640x640 10 generic roofs, 44.0ms
image 5/15 C:\Users\sdc02\aivle_school\deep_see_mini4-2\datasets\test\images\-2024-04-12-105929_png.rf.1362dd40b03edad75ad2dfcb21ad1c41.jpg: 640x640 6 generic roofs, 43.1ms
image 6/15 C:\Users\sdc02\aivle_school\deep_see_mini4-2\datasets\test\imag